# Compressing the K-cache: TurboQuant vs. JointQK vs. QPCA

A hands-on comparison of three quantization bases for transformer key caches, evaluated on the same LongBench-E Q/K/V data used in `longbench_data_tour.ipynb`. The mathematical setup follows `background/qpca_derivation_companion.html` — keep it open in another tab for the formal derivations.

## What we'll do

1. **Problem formulation** — what "compress K" actually optimises.
2. **Load the data** — same Qwen3-8B LongBench-E artefacts as `longbench_data_tour.ipynb`.
3. **Build per-(layer, kv-head) second moments** — $\Sigma_Q$, $\Sigma_K$ from `pooled_stats.pt`.
4. **Method 1 — TurboQuant** (called `v3` in the codebase, but it's the random-Hadamard / TurboQuant baseline).
5. **Method 2 — JointQK** — current production basis: orthogonal eigenvectors of $(\Sigma_Q \Sigma_K + \Sigma_K \Sigma_Q)/2$.
6. **Method 3 — QPCA** — closed-form optimum of the all-pairs inner-product objective.
7. **Define the four metrics** — `k_mse`, `logit_err`, `top-1`, `top-5`.
8. **Run all three methods** at $b \in \{2, 3, 4\}$ bits per coordinate.
9. **Side-by-side comparison** — who wins which metric, and why.

## Prerequisites

```bash
pip install torch huggingface_hub matplotlib
```

The notebook imports a few helpers from the `kvq` / `pipelines` packages in this repo; run from the repo root or set `PYTHONPATH` accordingly.

## 0. Setup — fresh environment in <2 minutes

Run these once on a clean machine. The notebook depends on the `kvq/` and `pipelines/` packages in this repo plus a handful of Python packages — no `pip install -e` needed.

### Prerequisites

- The repo cloned locally (gives you `kvq/`, `pipelines/`, `vendor/`, `_bootstrap.py`).
- [`uv`](https://docs.astral.sh/uv/) installed (or substitute `python -m venv` / `pip`).
- A GPU helps but isn't required (CPU ≈ 30 min; one A100 ≈ 3 min).
- Hugging Face Hub access for the 7 GB Q/K/V bundle (cached on first run).

### Commands — run from the repo root

```bash
# 1. Create a venv (Python 3.10+; tested with 3.12)
uv venv --python 3.12 .venv-qnb
source .venv-qnb/bin/activate

# 2. Install the notebook's dependencies (≈ 2 min, ~3 GB for torch + cuda libs)
uv pip install -r notebooks/requirements_quantization_notebook.txt

# 3. Register this venv as a Jupyter kernel
python -m ipykernel install --user --name=qnb --display-name="Quantization Notebook"

# 4. Launch — still from the repo root
jupyter lab notebooks/quantization_methods_comparison.ipynb
```

In Jupyter, select the **"Quantization Notebook"** kernel (Kernel ▸ Change Kernel ▸ Quantization Notebook), then run all cells top-to-bottom.

### Working directory

Launch Jupyter from **either the repo root or the `notebooks/` directory**. Cell §2 adds both `.` and `..` to `sys.path` and imports `_bootstrap.py` — whichever path is the actual repo root wins. (If you copied the notebook out of the repo, just move it back, or symlink `kvq/`, `pipelines/`, `vendor/`, and `_bootstrap.py` next to it.)

### What gets installed

`notebooks/requirements_quantization_notebook.txt` pins minimal versions of: `torch` (cu128 wheel — matches NVIDIA driver 570+ / CUDA 12.x), `numpy`, `scipy`, `einops`, `huggingface_hub`, `tqdm`, `matplotlib`, `ipykernel`, `jupyter`. **If your driver is ≥ 580** (CUDA 13), you can delete the `--extra-index-url` line and use any default torch wheel.

> **Why no `pip install -e .`** The notebook imports from `kvq/` and `pipelines/` via a `sys.path` bootstrap (`_bootstrap.py` in the repo root). Cell §2 below inserts the repo root into `sys.path` automatically — there is no editable-install step.

### If you hit problems

- **`ModuleNotFoundError: No module named 'kvq'`** — Jupyter wasn't launched from the repo root or `notebooks/`. `cd` into one of those and re-launch.
- **HuggingFace 401 / dataset not found** — `huggingface-cli login` before running cell §2 (dataset is `azaad/longbench-qkv-qwen3-small`).
- **`CUDA available? False` despite having a GPU** — your driver may be too old for the torch wheel. Run `nvidia-smi` to check; the cu128 wheel needs driver ≥ 570. If older, drop the `--extra-index-url` line and pin to `torch==2.5` (or wherever your driver supports).
- **OOM on shared GPU** — set `CUDA_VISIBLE_DEVICES=N` before launching Jupyter to pin to a free GPU, or `CUDA_VISIBLE_DEVICES=""` to fall back to CPU.

## 1. Problem formulation

We have queries $\{q_i\}_{i=1}^N \subset \mathbb{R}^d$ and keys $\{k_j\}_{j=1}^M \subset \mathbb{R}^d$ collected from a fixed transformer layer at a fixed (kv-)head. We compress each key to a $p$-dim code $r_j \in \mathbb{R}^p$ (often $p = d$ in our setting) and decode through a shared linear map $U \in \mathbb{R}^{d \times p}$. The objective is to preserve **every query–key inner product**:

$$L(U,\, \{r_j\}) \;=\; \sum_{i=1}^N \sum_{j=1}^M \bigl(q_i^\top k_j \;-\; q_i^\top U r_j\bigr)^2. \tag{1}$$

Expanding the square and exchanging the order of summation (companion §2):

$$L \;=\; \sum_j \bigl(k_j - U r_j\bigr)^\top M_q \bigl(k_j - U r_j\bigr), \qquad M_q \;:=\; \sum_i q_i q_i^\top.$$

The all-pairs loss collapses to a single $M_q$-weighted reconstruction objective on the keys; the query distribution enters only through its second-moment matrix $M_q$. Different choices of $U$ (and the bit allocation along its columns) define different quantization methods.

The three methods we compare all share the same scalar Lloyd–Max per-coordinate quantizer; they differ in:

| | basis $U$ | bit allocation score | comes from |
|---|---|---|---|
| **TurboQuant (v3)** | random orthogonal $R_{\text{rand}}$ | uniform | concentration of measure |
| **JointQK** | orthogonal eigvecs of $(\Sigma_Q \Sigma_K + \Sigma_K \Sigma_Q)/2$ | $q\_\text{diag}_i \cdot k\_\text{diag}_i$ | heuristic for logit variance |
| **QPCA** | $M_q^{-1/2} V$ where $V$ is eigvecs of $A = M_q^{1/2} \Sigma_K M_q^{1/2}$ | $\lambda_i$ (eigvals of $A$) | closed-form optimum of (1) |

## 2. Setup — load the LongBench Q/K/V data

We re-use the bundle from `longbench_data_tour.ipynb`. The `'small'` bundle (3 examples, ~7 GB) is enough for this notebook; switch to `'full'` for finer-grained statistics.

In [1]:
import json
import math
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from huggingface_hub import snapshot_download

# Make `kvq/` and `pipelines/` importable.
# The notebook lives at `<repo>/notebooks/...` — so the repo root is either CWD
# (jupyter launched from repo root) or CWD's parent (jupyter launched from notebooks/).
# Add both candidates to sys.path; whichever is the actual repo root wins.
sys.path[:0] = [str(Path.cwd().resolve()), str(Path.cwd().resolve().parent)]
import _bootstrap  # noqa: F401  — also adds vendor/kvpress to sys.path

# _bootstrap lives at the repo root — use its file location for downstream paths.
REPO = Path(_bootstrap.__file__).resolve().parent
print(f"Repo root: {REPO}")

# ----- Dataset configuration -----------------------------------------------
DATA_DIR = REPO / 'notebooks' / 'data'
DATASET  = 'small'

SPECS = {
    'small': {'dirname': 'query_stats_longbench_under4k_small',
              'repo_id': 'azaad/longbench-qkv-qwen3-small', 'approx_size': '~7 GB'},
    'full':  {'dirname': 'query_stats_longbench_under4k',
              'repo_id': 'azaad/longbench-qkv-qwen3-full',  'approx_size': '~57 GB'},
}
spec = SPECS[DATASET]
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = DATA_DIR / spec['dirname']

if not (DATA_ROOT / 'manifest.json').exists():
    print(f"Fetching {spec['repo_id']} ({spec['approx_size']})...")
    snapshot_download(repo_id=spec['repo_id'], repo_type='dataset', local_dir=str(DATA_ROOT))

manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
print(f"Using dataset '{DATASET}' at {DATA_ROOT}")
print(f"Model:   {manifest['config']['model']}")
print(f"Configs: {manifest['configs']}")
print(f"Examples: {len(manifest['examples'])}")

Repo root: /vault/amir/efficient-llm/teamily-project
Using dataset 'small' at /vault/amir/efficient-llm/teamily-project/notebooks/data/query_stats_longbench_under4k_small
Model:   Qwen/Qwen3-8B
Configs: ['qasper', 'hotpotqa', 'passage_retrieval_en']
Examples: 3


/vault/amir/efficient-llm/teamily-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Build per-(layer, kv-head) second moments

`pooled_stats.pt` stores `(mean, cov, second_moment)` per tensor name across all prefill tokens in the bundle. For our purposes we need:

- $\Sigma_Q$ — the **uncentered** second moment $\sum_i q_i q_i^\top$ per (layer, kv-head). Since Qwen3-8B uses GQA (32 Q-heads, 8 KV-heads, group size 4), we pool $\Sigma_Q$ across the 4 Q-heads inside each KV-head group, matching `pipelines/calibration/compute_stats.py`.
- $\Sigma_K$ — the uncentered second moment of keys per (layer, kv-head); already at the right shape.

The "uncentered" choice matches the QPCA derivation, which uses $M_q = \sum q_i q_i^\top$ (no subtraction of means).

In [ ]:
pooled = torch.load(DATA_ROOT / 'pooled_stats.pt', map_location='cpu', weights_only=False)

# Shapes from longbench_data_tour: q_post second_moment is (L, n_q_heads, d, d); k_post is (L, n_kv_heads, d, d).
q_second = pooled['q_post'][2]                  # (L, H_q, d, d), uncentered
k_second = pooled['k_post'][2]                  # (L, H_kv, d, d), uncentered

n_layers, n_q_heads, d_head, _ = q_second.shape
_, n_kv_heads, _, _ = k_second.shape
group_size = n_q_heads // n_kv_heads

# Pool Σ_Q across the GQA group: per kv_head h, sum over the 4 q_heads in [h*group:(h+1)*group].
sigma_q = q_second.reshape(n_layers, n_kv_heads, group_size, d_head, d_head).sum(dim=2)  # (L, H_kv, d, d)
sigma_k = k_second                                                                       # (L, H_kv, d, d)

print(f'n_layers = {n_layers}, n_q_heads = {n_q_heads}, n_kv_heads = {n_kv_heads}, head_dim = {d_head}')
print(f'GQA group size = {group_size}')
print(f'Σ_Q shape = {tuple(sigma_q.shape)}    Σ_K shape = {tuple(sigma_k.shape)}')

# Sanity: both matrices should be positive semidefinite per (L, h). Check the smallest eigenvalue on a sample.
L_check, H_check = 1, 0   # skip layer 0 (attention-sink layer, has different geometry)
e_q = torch.linalg.eigvalsh(sigma_q[L_check, H_check])
e_k = torch.linalg.eigvalsh(sigma_k[L_check, H_check])
print(f'\nAt (L={L_check}, h={H_check}):')
print(f'  Σ_Q eigvals  min={e_q.min().item():.3e}  max={e_q.max().item():.3e}  cond={(e_q.max()/e_q.clamp_min(1e-30).min()).item():.1e}')
print(f'  Σ_K eigvals  min={e_k.min().item():.3e}  max={e_k.max().item():.3e}  cond={(e_k.max()/e_k.clamp_min(1e-30).min()).item():.1e}')

## 4. Method 1 — TurboQuant

**TurboQuant** (Zandieh et al., *arXiv:2504.19874*, 2025; called `v3` in this codebase) is the **data-oblivious** baseline. Algorithm 1 of the paper:

$$\text{QUANT}_\text{mse}(k):\quad y = \Pi\,\frac{k}{\|k\|},\qquad \text{idx}_j = \arg\min_c\,|y_j - c|,$$

where $\Pi$ is a random orthogonal rotation and the codebook $\{c_1,\dots,c_{2^b}\} \subset [-1, 1]$ is optimised once against the Beta-distribution that rotated unit-vector coordinates follow in dimension $d$. Dequantisation rotates back and rescales by the stored $\|k\|$.

Three things to notice:

- **Per-vector L2 normalisation**: every key is rescaled to the unit sphere before rotation. The norm is stored separately (fp16) and reapplied at decode.
- **Single shared codebook**: one codebook is used for every coordinate (the Beta marginal is the same on each axis after a random rotation). No per-coord std.
- **Data-oblivious**: no $\Sigma_Q$, no $\Sigma_K$. The same rotation + codebook works on every dataset.

> The kvq codebase implements this in two equivalent classes: `vendor/turboquant-pytorch/compressors_v3.py:MSECompressor` (used by `TurboQuantPress` at bench time) and `kvq/compression/lloyd_max.py:Stage1MSECompressor` (used by `analyze_bases.py:empirical_v3_metrics` at calibration time). Both implement Algorithm 1 of the paper exactly; we use `Stage1MSECompressor` below because it has the cleaner `roundtrip(states)` interface for this notebook.

In [ ]:
from kvq.compression.lloyd_max import Stage1MSECompressor


class TurboQuantWrapper:
    """Adapt Stage1MSECompressor to the (T, d) -> (T, d) roundtrip interface used by score_example.

    Stage1MSECompressor expects (B, H, S, D); we wrap to accept (T, d) and de-wrap.
    Per production v3, the SAME random rotation is shared across all (layer, kv-head) —
    Algorithm 1 generates one Pi globally, not per head.
    """

    def __init__(self, head_dim, bits, seed=20260505, device='cpu'):
        self.tq = Stage1MSECompressor(head_dim=head_dim, bits=bits, seed=seed, device=device)
        self.head_dim = head_dim
        self.bits = bits
        # Expose the underlying rotation for inspection (the "basis" in our 5-stage taxonomy).
        self.forward_map = self.tq.Pi.T          # k @ forward = (Pi @ k_unit)^T row-vec convention
        self.inverse_map = self.tq.Pi            # codes @ inverse rotates back

    def to(self, device):
        self.tq.Pi = self.tq.Pi.to(device)
        self.tq.centroids = self.tq.centroids.to(device)
        self.tq.device = str(device)
        self.forward_map = self.tq.Pi.T
        self.inverse_map = self.tq.Pi
        return self

    def roundtrip(self, k):
        # k: (T, d) -> (T, d).  Stage1MSE handles per-vector normalize internally.
        out = self.tq.roundtrip(k.unsqueeze(0).unsqueeze(0))  # (1, 1, T, d)
        return out.squeeze(0).squeeze(0)


# Build one wrapper per bit width. Stage1MSECompressor.__init__ runs solve_lloyd_max(d, bits)
# the first time and caches the result, so subsequent constructions are cheap.
turbo_compressors = {
    bits: TurboQuantWrapper(head_dim=d_head, bits=bits, seed=20260505)
    for bits in [2, 3, 4]
}

for bits, comp in turbo_compressors.items():
    cents = comp.tq.centroids.cpu()
    print(f"TurboQuant b={bits}: {cents.numel()} centroids in [{cents.min():.4f}, {cents.max():.4f}]  "
          f"(Beta/Lloyd-Max for d={d_head}, data-oblivious)")

# Sanity: the rotation is orthogonal.
Pi = turbo_compressors[2].tq.Pi
err = (Pi @ Pi.T - torch.eye(d_head)).norm().item()
print(f"||Pi @ Pi^T - I|| = {err:.3e}  (should be ~0; Pi is orthogonal)")

# One Pi is shared across all (layer, kv-head) — this matches production v3.
print(f"Pi shape: {tuple(Pi.shape)}  (single rotation, NOT per (L, H))")

## 5. Method 2 — JointQK

**JointQK** uses the orthogonal eigenbasis of the symmetric cross-product:

$$R \;=\; \text{eigvecs}\!\left( \frac{\Sigma_Q \Sigma_K + \Sigma_K \Sigma_Q}{2} \right),$$

sorted by descending eigenvalue. This is the current **production K-side basis**. Two motivations:

- **Couples Q and K geometry**: directions where both $\Sigma_Q$ and $\Sigma_K$ have energy are emphasized.
- **Logit-variance bit allocation**: water-fill against the per-coord product
$$q\_\text{diag}_i \cdot k\_\text{diag}_i \;:=\; (R^\top \Sigma_Q R)_{ii} \cdot (R^\top \Sigma_K R)_{ii},$$
which, under a queries–keys independence approximation, equals the per-coord contribution to logit variance $\mathrm{Var}_{q,k}(\ell)$. Intuitively (companion §9 / Remark 9.3.1): more bits to coords that carry more logit information.

Heuristic (not closed-form optimal) for any specific objective, but empirically the production winner on attention-argmax retention.

In [ ]:
from pipelines.calibration.analyze_bases import jointqk_basis, regularize_batch

EPS = 1e-4

def build_jointqk_basis(sigma_q: torch.Tensor, sigma_k: torch.Tensor, eps: float = EPS) -> dict:
    """JointQK: orthogonal eigvecs of (Σ_Q Σ_K + Σ_K Σ_Q)/2.

    Bit-allocation score: per-coord diag(R^T Σ_Q R) · diag(R^T Σ_K R)  (logit variance approx).
    """
    R = jointqk_basis(sigma_q, sigma_k, eps=eps)  # (L, H, d, d), columns are eigvecs
    # forward applied as k @ R; inverse is R^T (orthogonal basis)
    forward = R
    inverse = R.transpose(-1, -2)
    q_diag = (inverse @ regularize_batch(sigma_q, eps) @ forward).diagonal(dim1=-2, dim2=-1).clamp_min(1e-30)
    k_diag = (inverse @ regularize_batch(sigma_k, eps) @ forward).diagonal(dim1=-2, dim2=-1).clamp_min(1e-30)
    score = q_diag * k_diag         # water-fill against logit-variance proxy
    std = k_diag.sqrt()
    return {'forward': forward, 'inverse': inverse, 'score': score, 'std': std}


jq = build_jointqk_basis(sigma_q, sigma_k)
print(f"JointQK basis:")
print(f"  forward {tuple(jq['forward'].shape)}, inverse {tuple(jq['inverse'].shape)}")
print(f"  score (q_diag · k_diag) at (L=1, H=0): top-5 = {jq['score'][1, 0].topk(5).values.tolist()}")

# Sanity: orthogonality of R.
err = (jq['forward'] @ jq['inverse'] - torch.eye(d_head)).norm(dim=(-2, -1)).max().item()
print(f"  ||F @ G - I||_F max over (L, H) = {err:.3e}")

## 6. Method 3 — QPCA

**QPCA** is the closed-form minimiser of the all-pairs inner-product loss (1), derived in companion §1–§6. Recipe:

1. Compute $A = M_q^{1/2} \Sigma_K M_q^{1/2}$.
2. Eigendecompose $A = V \Lambda V^\top$, sort eigenvalues descending.
3. Set:
   - **Forward** (encoder, applied to keys): $r_j \;=\; V^\top M_q^{1/2}\, k_j$
   - **Inverse** (decoder): $\hat k_j \;=\; M_q^{-1/2} V\, r_j$
   - **Bit allocation**: water-fill on $\Lambda$ alone.

The forward map is **non-orthogonal** in the standard Euclidean sense (companion Remark 7.3): $F^\top F \ne I$. Instead $U^* = M_q^{-1/2} V$ has $M_q$-orthonormal columns: $U^{*\top} M_q U^* = I$.

The clean structural property (companion Remark 6.4): the $M_q$-weighted key MSE after quantization equals the *plain Euclidean code MSE*. Each code coord contributes equally to the loss — that's why water-fill is on $\Lambda$ alone, not $\Lambda \cdot$ anything.

In [ ]:
def _sym(x: torch.Tensor) -> torch.Tensor:
    """Symmetrize, preserving dtype."""
    return 0.5 * (x + x.transpose(-1, -2))


def build_qpca_basis(sigma_q: torch.Tensor, sigma_k: torch.Tensor, eps: float = EPS) -> dict:
    """QPCA: closed-form optimum of all-pairs inner-product MSE.

    Forward = M_q^{1/2} V (applied as k @ forward on row-vectors).
    Inverse = V^T M_q^{-1/2} (so F @ G = I, with V eigvecs of A).
    Eigendecomps in fp64 for numerical safety (M_q^{-1/2} amplifies condition number).
    """
    sq = regularize_batch(sigma_q, eps).to(torch.float64)
    sk = regularize_batch(sigma_k, eps).to(torch.float64)

    # M_q^{1/2}, M_q^{-1/2} via spectral square root
    q_vals, q_vecs = torch.linalg.eigh(sq)
    q_vals = q_vals.clamp_min(float(eps))
    Mq_half     = _sym(q_vecs @ torch.diag_embed(q_vals.sqrt()) @ q_vecs.transpose(-1, -2))
    Mq_neg_half = _sym(q_vecs @ torch.diag_embed(q_vals.rsqrt()) @ q_vecs.transpose(-1, -2))

    # A = M_q^{1/2} Σ_K M_q^{1/2}, then eigendecompose
    A = _sym(Mq_half @ sk @ Mq_half)
    lam, V = torch.linalg.eigh(A)
    # Sort descending
    order = torch.argsort(lam, dim=-1, descending=True)
    lam = torch.gather(lam, -1, order)
    V = torch.gather(V, -1, order.unsqueeze(-2).expand(*V.shape[:-1], -1))

    forward = (Mq_half @ V).float()
    inverse = (V.transpose(-1, -2) @ Mq_neg_half).float()
    lam = lam.float().clamp_min(1e-30)
    return {'forward': forward, 'inverse': inverse, 'score': lam, 'std': lam.sqrt()}


qpca = build_qpca_basis(sigma_q, sigma_k)
print(f"QPCA basis:")
print(f"  forward {tuple(qpca['forward'].shape)}, inverse {tuple(qpca['inverse'].shape)}")
print(f"  eigenvalues Λ at (L=1, H=0): top-5 = {qpca['score'][1, 0].topk(5).values.tolist()}")

# Sanity: F @ G ≈ I (non-orthogonal but F G = I by construction).
err = (qpca['forward'] @ qpca['inverse'] - torch.eye(d_head)).norm(dim=(-2, -1)).max().item()
print(f"  ||F @ G - I||_F max over (L, H) = {err:.3e}  (should be tiny — F and G are inverses by construction)")

# F is NOT orthogonal — show that F^T F ≠ I.
err_orth = (qpca['forward'].transpose(-1, -2) @ qpca['forward'] - torch.eye(d_head)).norm(dim=(-2, -1)).max().item()
print(f"  ||F^T F - I||_F max  = {err_orth:.3e}  (non-zero — F is M_q-orthonormal, not Euclidean-orthonormal)")

## 7. Bit allocation and Lloyd–Max codebook

The downstream pipeline diverges slightly between **TurboQuant** and **JointQK / QPCA**:

- **TurboQuant** does not water-fill. Every coordinate gets exactly $b_\text{avg}$ bits because the rotated unit-vector marginals are identical in distribution (Beta-$(d)$), so no anisotropy to exploit. The shared codebook is the Beta-optimal Lloyd–Max codebook on $[-1, 1]$, computed once per $(d, b)$ pair.
- **JointQK and QPCA** water-fill against their per-coord score:

$$B_i \;\propto\; \tfrac{1}{2} \log_2 \mathrm{score}_i \;+\; \mathrm{const},\qquad \sum_i B_i = b_\text{avg} \cdot d,$$

then round (largest-remainder, preserves the sum) and clamp at 8 bits per coord ($2^8 = 256$ is already near-continuous for scalar Lloyd–Max). The resulting per-coord codebooks are unit-Gaussian centroids scaled by the method's per-coord std.

Both code paths land in a class that exposes `.roundtrip(k)`:

- **TurboQuant** → `Stage1MSECompressor` (in `kvq/compression/lloyd_max.py`) — the same class production `empirical_v3_metrics` uses.
- **JointQK / QPCA** → `PerCoordCompressor` (in `kvq/compression/per_coord.py`) with per-coord forward/inverse maps + per-coord bit budgets.

The visualisation below shows how each method spreads $b_\text{avg} = 3$ bits across the $d = 128$ coords (TurboQuant: trivial uniform; JointQK / QPCA: water-fill spread).

In [ ]:
from kvq.compression.per_coord import PerCoordCompressor
from pipelines.calibration.analyze_bases import allocate_bits

def build_compressors(basis: dict, k_bits: int, max_coord_bits: int = 8) -> dict:
    """For each (layer, kv-head), build a PerCoordCompressor (used by JointQK and QPCA)."""
    forward = basis['forward']    # (L, H, d, d)
    inverse = basis['inverse']
    score   = basis['score']      # (L, H, d)
    std     = basis['std']        # (L, H, d)
    bit_allocs = allocate_bits(score, k_bits, max_coord_bits=max_coord_bits).cpu()  # (L, H, d) int

    L, H = forward.shape[:2]
    comps = {}
    for l in range(L):
        for h in range(H):
            comps[(l, h)] = PerCoordCompressor(
                bits_per_coord=bit_allocs[l, h],
                std_per_coord=std[l, h],
                forward_map=forward[l, h],
                inverse_map=inverse[l, h],
            )
    return comps


# Bit-allocation histograms at b_avg=3.
b_demo = 3

# --- TurboQuant: uniform, no water-fill ---
print(f"{'TurboQuant':11s} bit allocation at b_avg={b_demo}:  uniform")
print(f"   {b_demo} bits: {d_head:3d} coords  " + '█' * d_head)
print(f"   (no water-fill — the shared Beta-optimal codebook has {2**b_demo} centroids in [-1, 1])\n")

# --- JointQK and QPCA: water-fill on each method's score ---
for name, basis in [('JointQK', jq), ('QPCA', qpca)]:
    alloc = allocate_bits(basis['score'], b_demo)  # (L, H, d)
    sample = alloc[1, 0]  # layer 1, kv_head 0
    counts = torch.bincount(sample, minlength=9)
    print(f"{name:11s} bit allocation at (L=1, H=0), b_avg={b_demo}:")
    for b in range(9):
        c = int(counts[b].item())
        bar = '█' * c
        print(f"   {b} bits: {c:3d} coords  {bar}")
    print()

## 8. Metrics

Following companion §8/§9, we compute four metrics per method per bit width. Each is averaged over (layers excluding 0) × (kv-heads) × (test examples).

| metric | formula | what it measures | who optimises it |
|---|---|---|---|
| **`k_mse`** | $\mathbb{E}\bigl[\,\lVert k - \hat k \rVert_2^2 \,\bigr]$ | unweighted key reconstruction error | plain PCA (neither of our 3 methods) |
| **`logit_err`** | $\mathbb{E}_{q,k}\bigl[\,(q^\top(k - \hat k))^2\,\bigr]$ | $M_q$-weighted key MSE (= all-pairs MSE up to a normalisation) | **QPCA, by construction** |
| **`top-1`** | $\mathbb{P}\bigl[\,\arg\max_t q^\top \hat k_t = \arg\max_t q^\top k_t\,\bigr]$ | attention argmax retention | nobody — empirical metric |
| **`top-5`** | $\mathbb{P}\bigl[\,\arg\max_t q^\top k_t \in \text{top-5}_t (q^\top \hat k_t)\,\bigr]$ | top-5 retention | nobody — empirical metric |

**Layer 0 is excluded** from headline aggregations because of anomalous attention-sink behaviour at the position-0 token (a stable finding from earlier in the project).

### Why we care about top-$k$: proxies for the post-softmax distribution

The four metrics aren't equally informative. `k_mse` and `logit_err` are *averages* over the population of $(q, k)$ pairs — they treat every pair equally. But attention doesn't:
$$\mathrm{output} \;=\; \mathrm{softmax}(q^\top K^\top)\, V \;=\; \sum_t \frac{\exp(q^\top k_t)}{\sum_{t'} \exp(q^\top k_{t'})}\, v_t.$$
The softmax **amplifies the largest logits exponentially**. In practice, attention distributions on real transformer prompts are highly concentrated — a handful of tokens carry most of the mass; for many heads/layers the effective support is single-digit. So whether downstream behaviour survives quantisation depends on whether *the few keys at the top* survive — not on the average MSE of all $T$ keys.

That makes **top-1 and top-5 retention direct proxies for the post-softmax distribution**:

- **`top-1` is a worst-case bound on argmax preservation.** If `top-1 = 0.9`, then on 90% of queries the dominant attention key after quantisation is the same as the original. When this fails, the softmax peak moves — the output context vector points at a *different* token's value. For sharply-peaked heads (think *attention sinks* or syntactic look-back heads), this is essentially the whole output.
- **`top-5` is a softer bound that allows the order within the top to scramble.** If `top-5 = 0.99`, the true top-1 key is still in the approximated *top-5* on 99% of queries — even if it's not the new top-1, softmax will still concentrate most mass somewhere near the right place. Empirically top-5 tracks attention-KL divergence closely on most heads.
- **MSE-flavoured metrics cannot replace top-$k$.** A quantiser that reduces `logit_err` by 50% on the bulk while *increasing* the perturbation on the small-margin top-1-vs-top-2 pairs would look better on `logit_err` and worse on `top-1` — exactly what QPCA does versus JointQK in the results below. The two answer different questions.

This is why production V3 reports and our calibration sweeps headline `top-1` and `top-5`: they're the cheapest computable quantities that bound the post-softmax behaviour, and they're sensitive to the *peaks* in a way `k_mse` and `logit_err` are not.

> *Why both `k_mse` and `logit_err`?* `logit_err` is the metric QPCA was designed to minimise, so we expect QPCA to win that one decisively. `k_mse` is the metric **none** of the methods targets — it's the unweighted key MSE that plain PCA would minimise. A near-tie on `k_mse` is a useful sanity signal: it means the three methods aren't trivially different (none of them is just unweighted PCA in disguise).

> *Why not directly measure attention KL?* It's a harder quantity to estimate stably with finite samples, and it requires the full softmax over $T \times T$. Top-1 / top-5 are cheap rank statistics that don't depend on a softmax temperature or numerical stability tricks. Companion §9 gives the formal margin-condition argument that ties argmax preservation to the actual attention output.

In [ ]:
def score_example(art: dict, comps_by_method: dict, k_bits_list: list) -> dict:
    """Score k_mse / logit_err / top-1 / top-5 on one example for all methods × bit widths.

    art:              dict from torch.load(examples/ex_XXX.pt), with q_post / k_post fp16 tensors.
    comps_by_method:  {method_name: {bits: {(l, h): PerCoordCompressor}}}.
    k_bits_list:      list of bit widths to evaluate.

    Returns nested dict accums[method][bits][layer] = {'mse_num', 'mse_den', 'logit_num', 'logit_den',
        'top1_num', 'top1_den', 'top5_num', 'top5_den'} (running sums for later aggregation).
    """
    q_all = art['q_post']    # (L, H_q, T, d) fp16
    k_all = art['k_post']    # (L, H_kv, T, d) fp16
    T = int(art['prompt_length'])
    L, H_kv = k_all.shape[0], k_all.shape[1]
    group_size = q_all.shape[1] // H_kv
    d = k_all.shape[-1]

    accums = {m: {b: {l: {'mse_num': 0.0, 'mse_den': 0,
                          'logit_num': 0.0, 'logit_den': 0,
                          'top1_num': 0, 'top1_den': 0,
                          'top5_num': 0, 'top5_den': 0}
                       for l in range(L)}
                   for b in k_bits_list}
              for m in comps_by_method}

    device = next(iter(next(iter(next(iter(comps_by_method.values())).values())).values())).forward_map.device
    for l in range(L):
        for h in range(H_kv):
            k = k_all[l, h, :T, :].to(device).float()    # (T, d)
            q = q_all[l, h*group_size:(h+1)*group_size, :T, :].to(device).float().reshape(-1, d)
            qq = q.transpose(0, 1) @ q                   # (d, d)
            k_t = k.transpose(0, 1)                      # (d, T)
            k_top5 = min(5, T)
            # Hoist computations that don't depend on the (method, bits) inner loop.
            real_logits = q @ k_t                        # (group*T, T)
            real_top = real_logits.argmax(dim=-1)        # (group*T,)
            n_q = int(real_top.numel())
            for method, comps_bits in comps_by_method.items():
                for bits in k_bits_list:
                    comp = comps_bits[bits][(l, h)]
                    k_hat = comp.roundtrip(k).float()
                    err = k - k_hat
                    acc = accums[method][bits][l]
                    acc['mse_num'] += float(err.square().sum().item())
                    acc['mse_den'] += int(err.numel())
                    ee = err.transpose(0, 1) @ err
                    acc['logit_num'] += float((qq * ee).sum().item())
                    acc['logit_den'] += int(q.shape[0] * T * T)
                    approx_logits = q @ k_hat.transpose(0, 1)
                    approx_top = approx_logits.argmax(dim=-1)
                    top1_match = int((real_top == approx_top).sum().item())
                    approx_top5 = approx_logits.topk(k_top5, dim=-1).indices
                    top5_match = int((approx_top5 == real_top.unsqueeze(-1)).any(dim=-1).sum().item())
                    acc['top1_num'] += top1_match
                    acc['top1_den'] += n_q
                    acc['top5_num'] += top5_match
                    acc['top5_den'] += n_q
    return accums


def merge_accums(a: dict, b: dict) -> dict:
    """Sum two accumulator nested dicts element-wise (used to pool across examples)."""
    for m in b:
        for bits in b[m]:
            for l in b[m][bits]:
                for k in b[m][bits][l]:
                    a[m][bits][l][k] = a.get(m, {}).get(bits, {}).get(l, {}).get(k, 0) + b[m][bits][l][k]
    return a


def finalize(accums: dict, exclude_layer_0: bool = True) -> dict:
    """Convert nested accumulators to per-method, per-bits scalar metrics."""
    out = {}
    for m, by_bits in accums.items():
        out[m] = {}
        for bits, by_layer in by_bits.items():
            layers = [l for l in by_layer if not (exclude_layer_0 and l == 0)]
            sums = {k: sum(by_layer[l][k] for l in layers)
                    for k in ['mse_num', 'mse_den', 'logit_num', 'logit_den',
                              'top1_num', 'top1_den', 'top5_num', 'top5_den']}
            out[m][bits] = {
                'k_mse':     sums['mse_num']   / max(1, sums['mse_den']),
                'logit_err': sums['logit_num'] / max(1, sums['logit_den']),
                'top1':      sums['top1_num']  / max(1, sums['top1_den']),
                'top5':      sums['top5_num']  / max(1, sums['top5_den']),
            }
    return out

## 9. Run the experiment

For each method, build compressors at $b \in \{2, 3, 4\}$ and score on every example in the bundle. Each example is a forward pass through Qwen3-8B on a single LongBench prompt; we score $L=36$ layers × $H_\text{kv}=8$ heads × 3 bit widths × 3 methods, which on the `small` bundle (3 examples) is a few minutes on CPU and a few seconds on GPU.

In [ ]:
K_BITS = [2, 3, 4]

# Pre-build all compressors: methods × bits × (L, H_kv)
print("Building compressors...")
comps_by_method = {}

# --- TurboQuant: ONE Stage1MSECompressor per bits, shared across all (L, H). ---
# This matches production v3: pipelines/calibration/analyze_bases.py:empirical_v3_metrics also
# instantiates one Stage1MSECompressor per bits and reuses it for every (layer, kv-head).
comps_by_method['TurboQuant'] = {
    bits: {(l, h): turbo_compressors[bits]
           for l in range(n_layers) for h in range(n_kv_heads)}
    for bits in K_BITS
}

# --- JointQK and QPCA: per-(L, H) PerCoordCompressor via build_compressors() ---
for name, basis in [('JointQK', jq), ('QPCA', qpca)]:
    comps_by_method[name] = {b: build_compressors(basis, k_bits=b) for b in K_BITS}

print(f"  TurboQuant: {len(K_BITS)} shared wrappers")
print(f"  JointQK + QPCA: 2 × {len(K_BITS)} × {n_layers * n_kv_heads} per-(L, H) compressors")

# Score all examples and pool.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")
if device.type == 'cuda':
    # Move all compressors to GPU. Each Stage1MSECompressor / PerCoordCompressor is tiny
    # (128x128 rotation + a small codebook). We do NOT preload full q_post/k_post to GPU —
    # each is ~1 GB at fp16; score_example moves per-(L, h) slices to GPU inside its loop.
    moved = set()
    for m in comps_by_method:
        for b in comps_by_method[m]:
            for k in comps_by_method[m][b]:
                comp = comps_by_method[m][b][k]
                if id(comp) not in moved:
                    comp.to(device)
                    moved.add(id(comp))

# Score across examples. Use `k_pooled` (NOT `pooled`) — `pooled` is the pooled_stats
# tensor dict loaded back in §3 and the V-side analysis needs it intact.
k_pooled = None
for i, entry in enumerate(manifest['examples']):
    print(f"  [{i+1}/{len(manifest['examples'])}] scoring {entry['file']}...", end=' ', flush=True)
    art = torch.load(DATA_ROOT / entry['file'], map_location='cpu', weights_only=False)
    a = score_example(art, comps_by_method, K_BITS)
    if k_pooled is None:
        k_pooled = a
    else:
        k_pooled = merge_accums(k_pooled, a)
    print('done')

print('\nFinalizing metrics (layer-0 excluded)...')
final = finalize(k_pooled, exclude_layer_0=True)
print('Done.')

## 10. Results

Side-by-side table, then a per-metric plot.

In [ ]:
# Pretty-print the comparison table.
print(f"{'method':<11} | {'b':<2} | {'top-1':>7} | {'top-5':>7} | {'k_mse':>11} | {'logit_err':>11}")
print("-" * 64)
for m in ['TurboQuant', 'JointQK', 'QPCA']:
    for b in K_BITS:
        d = final[m][b]
        print(f"{m:<11} | {b:<2} | {d['top1']:>7.4f} | {d['top5']:>7.4f} | "
              f"{d['k_mse']:>11.3e} | {d['logit_err']:>11.3e}")
    print()

# Per-(metric, bits) winner.
print("\nWinner per (metric, b):")
metrics_higher_better = {'top1', 'top5'}
for met in ['top1', 'top5', 'k_mse', 'logit_err']:
    cmp = max if met in metrics_higher_better else min
    print(f"  {met:<10}:", end='  ')
    for b in K_BITS:
        winner = cmp(final, key=lambda m: final[m][b][met])
        v = final[winner][b][met]
        marker = f"{v:.4f}" if met in metrics_higher_better else f"{v:.3e}"
        print(f"b={b}: {winner:<11} ({marker})", end='   ')
    print()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
metric_titles = {
    'top1':      ('top-1 retention', 'higher is better'),
    'top5':      ('top-5 retention', 'higher is better'),
    'k_mse':     ('k_mse  (unweighted key MSE)', 'lower is better'),
    'logit_err': ('logit_err  (Q-weighted key MSE)', 'lower is better'),
}
colors = {'TurboQuant': '#888', 'JointQK': '#328ac1', 'QPCA': '#1c5d2c'}
markers = {'TurboQuant': 's', 'JointQK': 'o', 'QPCA': '^'}

for ax, met in zip(axes, ['top1', 'top5', 'k_mse', 'logit_err']):
    for m in ['TurboQuant', 'JointQK', 'QPCA']:
        y = [final[m][b][met] for b in K_BITS]
        ax.plot(K_BITS, y, marker=markers[m], color=colors[m], label=m, linewidth=1.8, markersize=8)
    title, direction = metric_titles[met]
    ax.set_title(f"{title}\n({direction})", fontsize=11)
    ax.set_xlabel('bits per coord')
    ax.set_xticks(K_BITS)
    if met in ('k_mse', 'logit_err'):
        ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='best')
plt.tight_layout()
plt.show()

## 11. V-vector analysis

Now the value cache. V sits on the **output side** of attention,
$$\mathrm{output} \;=\; \mathrm{softmax}(qK^\top)\, V,$$
so V reconstruction matters through how it gets weighted-summed by the attention distribution — *not* through inner products with queries. There's no $M_q$ analog to absorb into the basis; the geometry is one-sided.

We compare two methods, each in **two variants** (centered vs uncentered) — four configurations total.

### The two methods

| method | basis | bit allocation | reference |
|---|---|---|---|
| **TurboQuant** | random orthogonal $\Pi$ (shared across $(L, h)$) | uniform $b_\text{avg}$ bits | Algorithm 1 of the paper, applied to V |
| **Eigenwaterfill** | eigvecs of the V-side second-moment matrix (per $(L, h)$) | water-fill on its eigenvalues | production `v_eigen_waterfill` |

### Centered vs uncentered

The Lloyd–Max codebook is designed for a **zero-mean** Gaussian source. V in transformer attention has non-trivial mean $\mu_V := \mathbb{E}[v]$ (unlike K, which is roughly mean-zero after RoPE). Two ways to handle this:

| variant | what it does | second-moment used |
|---|---|---|
| **centered** | subtract $\mu_V$ before quantising; add it back after | centered covariance $\mathrm{Cov}[V] = \mathbb{E}[(v - \mu)(v - \mu)^\top]$ |
| **uncentered** | quantise raw $v$ directly | uncentered second moment $\Sigma_V = \mathbb{E}[v v^\top] = \mathrm{Cov}[V] + \mu_V \mu_V^\top$ |

For eigenwaterfill, the choice of second-moment matters **structurally**: the basis (eigenvectors) differs between $\mathrm{Cov}[V]$ and $\Sigma_V$ — the latter has its top eigenvector pulled towards $\mu_V$, because the rank-1 perturbation $\mu_V \mu_V^\top$ dominates the spectrum when $\|\mu_V\|^2 \gg \mathrm{tr}(\mathrm{Cov}[V])/d$.

For TurboQuant, the basis is random regardless, but the per-vector L2 normalisation interacts with $\mu_V$ differently in the two variants:
- **Centered**: $\|v - \mu\| < \|v\|$ on average — the unit-norm representation has more diversity in direction.
- **Uncentered**: high-norm V's with $\|v\| \gg \|\mu\|$ look like unit vectors after normalisation; low-norm V's near $\mu$ get normalised to (roughly) $\mu / \|\mu\|$, losing all per-vector info.

### Hypothesis

Centered should beat uncentered for both methods. The gap should be larger for eigenwaterfill (where $\mu \mu^\top$ wastes a basis direction) than for TurboQuant (where per-vector normalisation partially absorbs the mean).

### Metric

$\mathrm{v\_mse} = \mathbb{E}[\|v - \hat v\|_2^2]$, layer-0 excluded — matching production `empirical_v_metrics` in `pipelines/calibration/analyze_bases.py`.

All compressors come from the production builder `build_v_compressor` in `kvq/compression/v_compressor_adapter.py`. Passing `mu_v_for_head=None` skips the `CenteredCompressor` wrapper (the uncentered variant); passing the actual mean enables centering.

In [ ]:
from kvq.compression.v_compressor_adapter import build_v_compressor

# pooled_stats stores (mean, centered_cov, uncentered_second_moment) per tensor name.
v_mean, v_cov, v_second = pooled['v']
print(f"V mean shape:          {tuple(v_mean.shape)}")
print(f"V centered cov shape:  {tuple(v_cov.shape)}")
print(f"V uncentered SM shape: {tuple(v_second.shape)}")

# Quick check at (L=1, H=0): how much of the uncentered SM mass is the rank-1 mu*mu^T?
mu_sample = v_mean[1, 0]
mu_outer_trace = float((mu_sample * mu_sample).sum())
cov_trace = float(v_cov[1, 0].diagonal().sum())
print(f"\nAt (L=1, H=0):")
print(f"  ‖μ_V‖²    = tr(μ μᵀ) = {mu_outer_trace:.4e}")
print(f"  tr(Cov[V])           = {cov_trace:.4e}")
print(f"  ratio                 = {mu_outer_trace / max(cov_trace, 1e-30):.2f}")
print(f"  → uncentered eigenwaterfill spends {mu_outer_trace/(mu_outer_trace+cov_trace)*100:.0f}% of variance on the μ-direction")

# Spectrum spread (useful sanity for water-fill — large dynamic range = more to gain).
e_cov = torch.linalg.eigvalsh(v_cov[1, 0])
e_sm  = torch.linalg.eigvalsh(v_second[1, 0])
print(f"\n  Cov[V] eigvals:  min={e_cov.min():.3e}, max={e_cov.max():.3e}, cond={e_cov.max()/e_cov.clamp_min(1e-30).min():.1e}")
print(f"  Σ_V    eigvals:  min={e_sm.min():.3e},  max={e_sm.max():.3e},  cond={e_sm.max()/e_sm.clamp_min(1e-30).min():.1e}")

V_BITS = [2, 3, 4]

# Four V configurations. Each entry: (kvq_method_name, second_moment_to_use, mu_to_use).
# - second_moment_to_use=None for v_turboquant (data-oblivious basis; pooled stats only feed mu).
# - mu_to_use=None ⇒ uncentered (no CenteredCompressor wrap).
v_variants = {
    'TurboQuant-centered':     ('v_turboquant',      None,     v_mean),
    'TurboQuant-uncentered':   ('v_turboquant',      None,     None),
    'Eigenwaterfill-centered': ('v_eigen_waterfill', v_cov,    v_mean),
    'Eigenwaterfill-uncentered':('v_eigen_waterfill', v_second, None),
}

print("\nBuilding V compressors (4 variants × 3 bits × 288 (L,H))...")
v_comps_by_method = {}
for nb_name, (kvq_name, sm_tensor, mu_tensor) in v_variants.items():
    v_comps_by_method[nb_name] = {}
    for bits in V_BITS:
        v_comps_by_method[nb_name][bits] = {}
        for l in range(n_layers):
            for h in range(n_kv_heads):
                cov_arg = None if sm_tensor is None else sm_tensor[l, h]
                mu_arg  = None if mu_tensor is None else mu_tensor[l, h]
                comp = build_v_compressor(
                    method=kvq_name,
                    cov_v_for_head=cov_arg,    # treated as the matrix to eigendecompose (Cov[V] or Σ_V)
                    head_dim=d_head,
                    bits=bits,
                    seed=20260505,
                    mu_v_for_head=mu_arg,      # None → no CenteredCompressor wrap
                )
                v_comps_by_method[nb_name][bits][(l, h)] = comp

n_unique = sum(len(set(id(c) for c in by_bits[b].values()))
               for by_bits in v_comps_by_method.values() for b in by_bits)
print(f"  Built {n_unique} unique compressor instances "
      f"({len(v_variants)} variants × {len(V_BITS)} bits × {n_layers * n_kv_heads} (L,H), "
      f"with TurboQuant sharing one rotation per bits-width)")

### V scoring kernel

Same loop as K, stripped down: no $q$ involved, only $v$. For each example, each $(L, h)$, each (variant, bits): roundtrip $v$, accumulate $\|v - \hat v\|^2$. The kernel doesn't know or care whether the compressor centers internally — `CenteredCompressor` handles that transparently on its `.roundtrip()`.

In [ ]:
def score_v_example(art, v_comps_by_method, v_bits_list, device):
    """Per-(variant, bits, layer) v_mse accumulator."""
    v_all = art['v']                                 # (L, H_kv, T, d)
    T = int(art['prompt_length'])
    L, H_kv = v_all.shape[0], v_all.shape[1]
    accums = {m: {b: {l: {'mse_num': 0.0, 'mse_den': 0} for l in range(L)}
                  for b in v_bits_list} for m in v_comps_by_method}

    for l in range(L):
        for h in range(H_kv):
            v = v_all[l, h, :T, :].to(device).float()
            for method, by_bits in v_comps_by_method.items():
                for bits in v_bits_list:
                    comp = by_bits[bits][(l, h)]
                    v_hat = comp.roundtrip(v).float()
                    err = v - v_hat
                    acc = accums[method][bits][l]
                    acc['mse_num'] += float(err.square().sum().item())
                    acc['mse_den'] += int(err.numel())
                    del v_hat, err
            del v
    return accums


def finalize_v(accums, exclude_layer_0=True):
    out = {}
    for m, by_bits in accums.items():
        out[m] = {}
        for bits, by_layer in by_bits.items():
            layers = [l for l in by_layer if not (exclude_layer_0 and l == 0)]
            num = sum(by_layer[l]['mse_num'] for l in layers)
            den = sum(by_layer[l]['mse_den'] for l in layers)
            out[m][bits] = num / max(1, den)
    return out


# Move V compressors to GPU (dedupe shared instances via id()).
print(f"Moving V compressors to {device}...")
v_moved = set()
for m in v_comps_by_method:
    for b in v_comps_by_method[m]:
        for k in v_comps_by_method[m][b]:
            comp = v_comps_by_method[m][b][k]
            if id(comp) not in v_moved:
                comp.to(device)
                v_moved.add(id(comp))

# Score across all examples.
v_pooled = None
for i, entry in enumerate(manifest['examples']):
    print(f"  [{i+1}/{len(manifest['examples'])}] V-scoring {entry['file']}...", end=' ', flush=True)
    art = torch.load(DATA_ROOT / entry['file'], map_location='cpu', weights_only=False)
    a = score_v_example(art, v_comps_by_method, V_BITS, device)
    if v_pooled is None:
        v_pooled = a
    else:
        for m in a:
            for b in a[m]:
                for l in a[m][b]:
                    for k in a[m][b][l]:
                        v_pooled[m][b][l][k] += a[m][b][l][k]
    print('done')

v_final = finalize_v(v_pooled, exclude_layer_0=True)

variant_order = ['TurboQuant-centered', 'TurboQuant-uncentered',
                 'Eigenwaterfill-centered', 'Eigenwaterfill-uncentered']

print()
print(f"{'variant':<28} | {'b':<2} | {'v_mse':>12}")
print('-' * 50)
for m in variant_order:
    for b in V_BITS:
        print(f"{m:<28} | {b:<2} | {v_final[m][b]:>12.4e}")
    print()

# Per-bits ranking
print("Ranking per bits (lower v_mse = better):")
for b in V_BITS:
    ranked = sorted(variant_order, key=lambda m: v_final[m][b])
    line = ', '.join(f"{m}={v_final[m][b]:.3e}" for m in ranked)
    print(f"  b={b}: {line}")

# Centering effect: how much does centering help each method?
print("\nCentering effect (ratio uncentered/centered, > 1 = centering wins):")
for method in ('TurboQuant', 'Eigenwaterfill'):
    for b in V_BITS:
        c = v_final[f'{method}-centered'][b]
        u = v_final[f'{method}-uncentered'][b]
        ratio = u / max(c, 1e-30)
        print(f"  {method:<16} b={b}:  centered={c:.3e},  uncentered={u:.3e},  ratio={ratio:.2f}x")

In [ ]:
# V-side plot — group by method, distinguish centered/uncentered with marker style.
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0))

style = {
    'TurboQuant-centered':       dict(color='#888',    marker='s', linestyle='-',  label='TurboQuant · centered'),
    'TurboQuant-uncentered':     dict(color='#888',    marker='s', linestyle='--', label='TurboQuant · uncentered', markerfacecolor='none'),
    'Eigenwaterfill-centered':   dict(color='#1c5d2c', marker='^', linestyle='-',  label='Eigenwaterfill · centered'),
    'Eigenwaterfill-uncentered': dict(color='#1c5d2c', marker='^', linestyle='--', label='Eigenwaterfill · uncentered', markerfacecolor='none'),
}
for m in variant_order:
    y = [v_final[m][b] for b in V_BITS]
    ax.plot(V_BITS, y, linewidth=1.8, markersize=8, **style[m])

ax.set_title('V-side reconstruction — v_mse  (lower is better)', fontsize=11)
ax.set_xlabel('bits per coord')
ax.set_ylabel('v_mse')
ax.set_xticks(V_BITS)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9, loc='best')
plt.tight_layout()
plt.show()

## 12. Interpretation — why JointQK beats QPCA on argmax

The full discussion is in `qpca_derivation_companion.html` §9. Three independent mechanisms:

1. **M1 — Argmax is a threshold statistic.** Top-1 preservation depends on a margin condition ($\Delta\ell_{t^*} - \Delta\ell_t > -m(t^*, t)$), which is determined by the *joint* distribution of $(m_t, \sigma_t)$. MSE is a single moment of the marginals — it cannot bound argmax survival.

2. **M2 — The Linder–Zamir–Zeger optimality doesn't extend to argmax.** LZZ proves QPCA optimal in the high-resolution limit for smooth, locally quadratic distortion measures. Argmax is a 0/1 threshold — outside the LZZ class.

3. **M3 — The bit allocations differ in what they preserve.** QPCA water-fills on $\lambda_i$ alone (equalises per-coord MSE in code space). JointQK water-fills on $q\_\text{diag}_i \cdot k\_\text{diag}_i$ — the per-coord logit variance under independence. Argmax is determined by the few coords with the largest logits; JointQK over-budgets exactly those coords.

**The smoking gun**: across $b \in \{2, 3, 4\}$, QPCA wins `logit_err` (its design objective) but ties with JointQK on `k_mse` (which neither targets). This rules out implementation error — both methods are doing exactly what their respective theories say they should. The argmax deficit is a *loss-mismatch* phenomenon, not a basis-construction error.

### A clean intuition

**Argmax is a conjunctive event.** It survives iff the gap $q^\top(k_{\text{top}} - k_{\text{runner}})$ is preserved — which depends on directions where *both* the query has energy *and* the keys differ. JointQK's bit-allocation score $q\_\text{diag}_i \cdot k\_\text{diag}_i$ is **literally that conjunction** (a product of marginal energies). QPCA's score $\lambda_i$ is the eigenvalue of a *coupled* quadratic form $A = M_q^{1/2} \Sigma_K M_q^{1/2}$ — large in directions where the cross-coupling lifts it, even when neither marginal is individually large.

**Products favor argmax; couplings favor averages.** In the commuting limit ($\Sigma_Q \Sigma_K = \Sigma_K \Sigma_Q$), the two methods are mathematically identical. The empirical gap is the price of the non-commuting structure that's inherent to LLM attention (Q and K come from different projection matrices).

### What this implies

- **For deployment**: stick with JointQK. QPCA's optimality is real but for the wrong loss.
- **For theory**: a useful next basis design should target a *margin-aware* objective — e.g., top-$k$ attention-KL, or an explicit margin objective. QPCA is now the closed-form baseline that any margin-aware basis must beat.